<h1 style="color: #25a890; font-size: 2.2em; margin-bottom: 5px; font-weight: 700;">Customer Segmentation | Association</h1>

<p style="margin: 3px 0; font-size: 1.1em; font-weight: 600;">NOVA IMS <span style="font-weight: 300;">| Machine Learning II</span></p>
<p style="margin: 3px 0; font-size: 0.95em;"><strong">Professors</strong> Fernando Bação & Ivo Bernardo</p>


<br>
<table style="width: 100%; border-collapse: collapse; font-size: 1em;">
    <tr>
        <td style="padding: 4px 0; font-weight: 500;">Diogo Gonçalves</td>
        <td style="padding: 4px 0; text-align: right; font-family: monospace;">(20241817)</td>
    </tr>
    <tr>
        <td style="padding: 4px 0; font-weight: 500;">Gustavo Franco</td>
        <td style="padding: 4px 0; text-align: right; font-family: monospace;">(20241806)</td>
    </tr>
    <tr>
        <td style="padding: 4px 0; font-weight: 500;">Simão Costa</td>
        <td style="padding: 4px 0; text-align: right; font-family: monospace;">(20241772)</td>
    </tr>


    


<h3 style="color: #25a890; font-size: 1.3em; font-weight: 600; margin-bottom: 10px;">Table of Contents</h3>
<div style="font-family: Arial, sans-serif; line-height: 1.8; margin-bottom: 30px; padding-left: 10px;">
    <ol style="margin-top: 0; padding-left: 20px;">
        <li><a href="#library-imports" style="color: #25a890; text-decoration: none; font-weight: 500;">Library Imports</a></li>
        <li><a href="#data-import" style="color: #25a890; text-decoration: none; font-weight: 500;">Data Import</a></li>
        <li><a href="#association-rules" style="color: #25a890; text-decoration: none; font-weight: 500;">Association Rules</a></li>
        <li><a href="#results" style="color: #25a890; text-decoration: none; font-weight: 500;">Results</a></li>
        <li><a href="#cluster-campaigns" style="color: #25a890; text-decoration: none; font-weight: 500;">Cluster Campaigns</a></li>
    </ol>
</div>

<h2 style="color: #25a890; margin-bottom: 5px; font-weight: 700;">Library Imports</h2>

In [1]:
#pip install mlxtend --upgrade

In [2]:
import ast
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

<h2 style="color: #25a890; margin-bottom: 5px; font-weight: 700;">Data Import</h2>

In [3]:
basket = pd.read_csv('customer_basket.csv')
clusters = pd.read_csv('cluster_assignments.csv')
basket = basket.merge(clusters, on='customer_id')
basket['cluster'] = basket['cluster'].astype(int)

print(f"Total baskets: {len(basket)}")
print(f"Baskets per cluster:\n{basket['cluster'].value_counts().sort_index()}")

Total baskets: 100000
Baskets per cluster:
cluster
0     9488
1    31595
2    37892
3    14505
4     6520
Name: count, dtype: int64


In [4]:
basket.head(5)

,invoice_id,list_of_goods,customer_id,cluster
0,3700630,"['chicken', 'rice', 'pepper', 'whole wheat ric...",12912,1
1,10242376,"['low fat yogurt', 'tomatoes', 'pepper', 'aspa...",22853,3
2,91550,"['cake', 'tomatoes', 'pancakes', 'iPad', 'fina...",19,2
3,3137503,"['cereals', 'megaman zero', 'final fantasy XIX...",10995,0
4,7165061,"['rice', 'frozen smoothie', 'black tea', 'tea'...",27807,0


<h2 style="color: #25a890; margin-bottom: 5px; font-weight: 700;">Association Rules</h2>

<h3 style="color: #25a890; font-size: 1.3em; font-weight: 600; margin-bottom: 10px;"> Transform the data</h3>

In [5]:
# Convert String Lists to Real Lists
basket['list_of_goods'] = basket['list_of_goods'].apply(ast.literal_eval)

print(f"Sample cluster: {basket['cluster'].iloc[0]}")
print(f"Sample basket: {basket['list_of_goods'].iloc[0]}")

Sample cluster: 1
Sample basket: ['chicken', 'rice', 'pepper', 'whole wheat rice', 'shrimp', 'toothpaste', 'gums', 'cereals', 'bluetooth headphones', 'vacuum cleaner', 'body spray']


In [6]:
def get_association_rules(transactions, min_support=0.02, min_confidence=0.2):
    te = TransactionEncoder()
    te_array = te.fit_transform(transactions)
    df = pd.DataFrame(te_array, columns=te.columns_)
    frequent_itemsets = apriori(df, min_support=min_support, use_colnames=True)
    if frequent_itemsets.empty:
        print("No frequent itemsets found")
        return pd.DataFrame()
    rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
    return rules.sort_values('lift', ascending=False)

# Convert transactions to one-hot encoding
# Generate frequent itemsets using Apriori with a minimum support threshold
# Derive association rules using lift >= 1
# Return rules sorted by descending lift

In [7]:
all_products = [item for sublist in basket['list_of_goods'] for item in sublist]
print(f"Unique products: {len(set(all_products))}")
print(f"Most common products:\n{pd.Series(all_products).value_counts().head(10)}")

Unique products: 164
Most common products:
asparagus       12811
airpods         12145
cereals          9952
fresh bread      9934
butter           9654
eggs             9241
protein bar      8695
cooking oil      8623
toilet paper     8395
babies food      8318
Name: count, dtype: int64


<h3 style="color: #25a890; font-size: 1.3em; font-weight: 600; margin-bottom: 10px;"> Check Missing Values</h3>

In [ ]:
# (0,4) -> 0 missing values in the 5 (0, 1, 2, 3, 4) clusters
print(basket[basket['cluster'].isna()].shape)
basket = basket.dropna(subset=['cluster'])
print(basket['cluster'].value_counts().sort_index())

(0, 4)
cluster
0     9488
1    31595
2    37892
3    14505
4     6520
Name: count, dtype: int64


<h3 style="color: #25a890; font-size: 1.3em; font-weight: 600; margin-bottom: 10px;"> Cluster Rules</h3>

In [9]:
cluster_rules = {}
for cluster_id in sorted(basket['cluster'].unique()):
    transactions = basket[basket['cluster'] == cluster_id]['list_of_goods'].tolist()
    rules = get_association_rules(transactions)
    cluster_rules[cluster_id] = rules

# Dictionary to store association rules per cluster
# Iterate over each cluster in sorted order
# Extract transactions belonging to the current cluster
# Generate association rules for that cluster
# Store the rules indexed by cluster ID

In [ ]:
for cluster_id in sorted(basket['cluster'].unique()):
    print(f"\n{'='*50}")
    print(f"CLUSTER {cluster_id} — Top Unique Rules by Lift")
    print(f"{'='*50}")
    rules = cluster_rules[cluster_id]
    if not rules.empty:
        rules = rules.copy()
        rules['pair'] = rules.apply(lambda row: tuple(sorted([str(sorted(row['antecedents'])), str(sorted(row['consequents']))])), axis=1)
        unique_rules = rules.drop_duplicates(subset='pair')
        top10 = unique_rules.nlargest(10, 'lift')[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
        display(top10)

# Iterate through clusters and print results per cluster
# Retrieve association rules computed for the current cluster
# Create a canonical (order-independent) rule pair to remove symmetric duplicates
# Keep only unique antecedent–consequent pairs
# Select and display the top 10 rules with highest lift


CLUSTER 0 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
577,"(green tea, eggs)",(cereals),0.021185,0.483173,1.368053
579,(green tea),"(cereals, eggs)",0.021185,0.196673,1.361076
581,(eggs),"(green tea, cereals)",0.021185,0.061865,1.349372
542,"(butter, oatmeal)",(milk),0.020341,0.260811,1.344146
610,(cereals),"(salt, eggs)",0.026454,0.074903,1.338380
603,(eggs),"(oil, cereals)",0.021606,0.063096,1.330351
774,"(honey, fresh bread)",(tea),0.022028,0.272846,1.330299
675,(honey),"(cereals, oatmeal)",0.022344,0.109959,1.329027
684,"(milk, cereals)",(oatmeal),0.021185,0.271989,1.328853
672,"(honey, cereals)",(oatmeal),0.022344,0.270754,1.322816



CLUSTER 1 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
11,(shower gel),(deodorant),0.024877,0.302191,3.445593
17,(shower gel),(shampoo),0.024782,0.301038,3.441135
23,(tooth brush),(shower gel),0.024276,0.274026,3.328668
8,(shampoo),(deodorant),0.024846,0.284009,3.238273
18,(shampoo),(tooth brush),0.024687,0.282200,3.185459
24,(toothpaste),(shower gel),0.024687,0.252672,3.069276
12,(deodorant),(tooth brush),0.023675,0.269939,3.047057
26,(toothpaste),(tooth brush),0.025859,0.264658,2.987452
21,(shampoo),(toothpaste),0.025162,0.287627,2.943817
14,(toothpaste),(deodorant),0.024403,0.249757,2.847735



CLUSTER 2 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
2,(airpods),(energy drink),0.023884,0.200044,2.612018
13,(napkins),(cooking oil),0.020295,0.228733,2.583352
1,(bluetooth headphones),(airpods),0.022327,0.297468,2.491528
14,(dog food),(napkins),0.022274,0.215581,2.429744
4,(babies food),(cooking oil),0.021720,0.214100,2.418084
8,(babies food),(napkins),0.021588,0.212799,2.398390
6,(dog food),(babies food),0.023963,0.231928,2.286221
10,(dog food),(cooking oil),0.020506,0.198467,2.241528



CLUSTER 3 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
177,"(asparagus, carrots)",(salad),0.027646,0.438251,3.555277
159,"(asparagus, avocado)",(salad),0.026680,0.432886,3.511751
158,"(asparagus, salad)",(avocado),0.026680,0.407368,3.465618
152,"(asparagus, carrots)",(avocado),0.025577,0.405464,3.449421
198,(salad),"(asparagus, spinach)",0.027094,0.219799,3.409818
164,"(asparagus, spinach)",(avocado),0.025440,0.394652,3.357439
204,(salad),"(asparagus, tomatoes)",0.026405,0.214206,3.326612
175,(avocado),"(asparagus, tomatoes)",0.024957,0.212317,3.297274
187,(spinach),"(asparagus, carrots)",0.027508,0.206843,3.278969
79,(energy drink),(bluetooth headphones),0.025577,0.280423,3.269726



CLUSTER 4 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
1,(asparagus),(airpods),0.021779,0.137597,1.059187


<h2 style="color: #25a890; margin-bottom: 5px; font-weight: 700;">Results</h2>

<p style="color: #25a890; font-size: 0.9em;">
    <strong style="color: #25a890;">Support</strong> - proportion of baskets that contain both items
    <br><br>
    <strong style="color: #25a890;">Confidence</strong> - percentage of bought together ('If someone buys item A, we have a x% confidence that buys item B)
    <br><br>
    <strong style="color: #25a890;">Lift</strong> - measure of how much more likely someone is to buy two items together compared to independently
    <ul style="color: #25a890;">
        <li>Lift = 1 → no association, purely coincidental</li>
        <li>Lift > 1 → positive association, bought together more than expected</li>
        <li>Lift < 1 → negative association, rarely bought together</li>
    </ul>
    <br>
    <strong style="color: #25a890;">Cluster 0 - "Big Family Shopper"</strong>
    <ul style="color: #25a890;">
        <li>Rules are centered around a breakfast basket: cereals ↔ green tea ↔ eggs appear together consistently (lift ~1.35–1.37), and oatmeal + honey cluster around cereals and milk (lift ~1.32–1.34)</li>
        <li>Lifts are modest (1.3–1.4) but very consistent — this is a stable, habitual shopping pattern rather than a strong impulse association</li>
        <li>Support is low (~0.021) which reflects the large cluster size — even niche combos appear frequently in absolute terms</li>
    </ul>
    <strong style="color: #25a890;">Cluster 1 - "New Customers + Tech Enthusiast"</strong>
    <ul style="color: #25a890;">
        <li>Dominated entirely by hygiene products: deodorant, shower gel, shampoo, toothbrush and toothpaste all cross-associate with lifts between 2.85 and 3.45</li>
        <li>These are the strongest and most concentrated rules across all clusters — customers in this cluster buy hygiene products as a complete set, not individually</li>
        <li>High confidence values (0.25–0.30) reinforce this: if someone buys shampoo, there's a ~30% chance shower gel is in the same basket, which is very high for grocery data</li>
    </ul>
    <strong style="color: #25a890;">Cluster 2 - "Most Loyal"</strong>
    <ul style="color: #25a890;">
        <li>Top rule: airpods → energy drink (lift 2.61) and bluetooth headphones → airpods (lift 2.49) — tech items co-occur strongly</li>
        <li>Interestingly, household/family items also appear: napkins ↔ cooking oil (lift 2.58), dog food ↔ napkins (lift 2.43), babies food ↔ cooking oil (lift 2.42) — suggesting this cluster mixes loyal tech buyers with family shoppers</li>
        <li>The two sub-patterns (tech and household) don't overlap, which may indicate two distinct sub-profiles within this cluster worth noting as a limitation</li>
    </ul>
    <strong style="color: #25a890;">Cluster 3 - "Bargain Hunter"</strong>
    <ul style="color: #25a890;">
        <li>Strongest rules overall: asparagus, carrots, salad, avocado and spinach form a tightly connected fresh vegetable network with lifts consistently above 3.2</li>
        <li>The highest rule (asparagus + carrots → salad, lift 3.56) means customers who buy asparagus and carrots are 3.56× more likely to also buy salad than by chance — a very actionable cross-sell signal</li>
        <li>The only outlier is energy drink → bluetooth headphones (lift 3.27), which likely reflects a small but consistent sub-segment within the cluster</li>
    </ul>
    <strong style="color: #25a890;">Cluster 4 - "Biggest Buyers"</strong>
    <ul style="color: #25a890;">
        <li>Only 1 rule found: asparagus → airpods (lift 1.06) — this is essentially lift ≈ 1, meaning no real association exists</li>
        <li>This is expected for the "Biggest Buyers" cluster: they buy everything, so no specific item pairs stand out above random chance</li>
        <li>Association rules are not useful here — campaign design for this cluster should rely on spending profile and basket size instead</li>
    </ul>
</p>

<h2 style="color: #25a890; margin-bottom: 5px; font-weight: 700;">Cluster Campaigns</h2>

<p style="color: #25a890; font-size: 0.9em;">
    <strong>Cluster 0 - "Big Family Shopper"</strong>
    <ul style="color: #25a890;">
        <li>Campaign: <strong>"Family Breakfast Bundle - buy cereals + green tea + eggs and get honey and outmeal 30% off"</strong>, <strong>"Party Ready Bundle - buy 3 alcoholic drinks and get 20% off non-alcoholic drinks"</strong> and <strong>"Family Hygiene Pack - buy €30 in hygiene products and get the cheapest one free"</strong></li>
    </ul>
    <strong style="color: #25a890;">Cluster 1 - "New Customers + Tech Enthusiast"</strong>
    <ul style="color: #25a890;">
        <li>Campaign: <strong>"Hygiene Kit - buy shampoo + shower gel + deodorant and get a toothbrush free"</strong>, <strong>"Fresh Tech Deal - buy €50 in fresh food and get 10% off any electronics"</strong> and <strong>"Welcome to the Club - sign up for loyalty card and get 15% off in Hygiene/Tech products"</strong> (category changes every week)</li>
    </ul>
    <strong style="color: #25a890;">Cluster 2 - "Most Loyal"</strong>
    <ul style="color: #25a890;">
        <li>Campaign: <strong>"Tech Bundle - buy AirPods and get a free Energy Drink"</strong> or <strong>"Loyal Member Reward - use your loyalty card and get 10% off everything from the category Fresh Food/Hygiene/Tech"</strong> (category changes every week)</li>
    </ul>
    <strong style="color: #25a890;">Cluster 3 - "Bargain Hunter"</strong>
    <ul style="color: #25a890;">
        <li>Campaign: <strong>"Healthy Plate Deal - buy asparagus + salad + spinach and get carrots + avocado at half price"</strong> and <strong>"Double Promotion Day — every Wednesday all your usual promoted items get an extra 10% off"</strong></li>
    </ul>
    <strong style="color: #25a890;">Cluster 4 - "Biggest Buyers"</strong>
    <ul style="color: #25a890;">
        <li>Campaign: <strong>"VIP Exclusive — spend over €150 and get 15% off your entire basket"</strong></li>
    </ul>
</p>